# Seq2Seq 모델 구축 튜토리얼

## 1. 인코더 구현

입력 문장을 받아서 LSTM 돌려서 hidden state, cell state 뽑음. 이 hidden이 바로 디코더한테 넘겨줄 문장의 요약본임.

구성 임베딩 레이어 하나로 단어 인덱스(정수)를 벡터로 바꿔주고, LSTM이 그 벡터들을 시간순으로 처리해서 outputs랑 hidden, cell 뽑아냄.

여기서 batch_first=True는 입력 shape이 (batch, seq_len, feature) 순서라는 뜻. 기본값은 (seq_len, batch, feature)인데 직관 안맞아서 보통 True로 둠

In [1]:
import torch.nn as nn

# 인코더 클래스 시작
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        # input_dim = 사전크기 (vocab_size랑 같은거임)
        # emb_dim = 단어 한개 표현하는 벡터 차원
        # hidden_dim = lstm 메모리셀 크기
        super().__init__()

        # 단어 인덱스를 벡터로 바꿔주는 룩업테이블
        self.embedding = nn.Embedding(input_dim, emb_dim)
        
        # 시퀀스 처리하는 lstm 본체
        # batch_first 켜놨으니까 입력은 (batch, seq, emb_dim) 순서
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        # src shape: (batch, seq_len) <- 단어 인덱스들
        print("입력 Shape:", src.size())

        # 임베딩 통과 -> (batch, seq_len, emb_dim)
        embedded = self.embedding(src)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        # lstm 통과
        # outputs = 매 timestep마다의 hidden 모음
        # h_0, c_0 = 마지막 timestep의 hidden, cell state (이게 디코더로 넘어감)
        outputs, (h_0, c_0) = self.rnn(embedded)
        print("LSTM Layer의 Output Shape:", outputs.size())
        print("LSTM Layer의 Hidden State Shape:", h_0.size())
        print("LSTM Layer의 Cell State Shape:", c_0.size())

        # 셋다 리턴해줌. h_0이 사실상 context vector 역할
        return outputs, h_0, c_0

print("완료")

완료


## 2. 하이퍼파라미터 세팅

모델 돌려볼라고 숫자 몇개 정하는 단계. 실제론 데이터 보고 결정해야되지만 지금은 그냥 동작 확인용이라 적당히 박음.

vocab_size는 모델이 아는 단어 총 개수, 그러니까 사전 두께임. emb_size는 단어 하나를 몇차원 벡터로 표현할지 정하는거고 단어 한개를 설명하는데 쓰는 형용사 개수라고 보면 됨. lstm_size는 lstm 내부 hidden state 크기인데 모델 단기기억 용량이라고 생각하면 편함. batch_size는 한번에 몇 문장 처리할지, sample_seq_len은 그냥 테스트용 문장 길이라 짧게 3개로 둠

In [2]:
# 사전크기 3만개로 잡음 (보통 한국어/영어 모델에서 흔한 크기)
vocab_size = 30000

# 단어 한개를 256차원 벡터로 압축
emb_size = 256

# lstm 메모리 512짜리. 너무 크면 학습 느리고 작으면 표현력 딸림
lstm_size = 512

# 일단 한문장씩만 봐볼거라 1
batch_size = 1

# 테스트용 문장 길이 (단어 3개짜리 문장 하나 만들거임)
sample_seq_len = 3

# 확인차 출력
print("Vocab Size: {0}".format(vocab_size))
print("Embedidng Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedidng Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



## 3. 인코더 동작 확인

위에서 만든 인코더에 가짜 데이터 넣어보고 shape 잘 나오나 보자. torch.randint로 0부터 vocab_size 사이 랜덤 정수 뽑아서 가짜 문장 만드는거임. 어차피 학습 안할거라 뜻은 의미없고 텐서모양만 맞으면 됨.

예상되는 흐름은 입력이 (1, 3)으로 들어가서 임베딩 거치면 (1, 3, 256)이 되고, lstm 통과하면 output은 (1, 3, 512), hidden이랑 cell은 (1, 1, 512)로 나와야됨

In [3]:
import torch

# 인코더 인스턴스 생성. 위에서 정한 하이퍼파라미터 그대로 박아넣음
encoder = Encoder(vocab_size, emb_size, lstm_size)

# 가짜 입력문장 만들기. 0~30000 사이 랜덤 인덱스 (1, 3)짜리
sample_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

# 인코더 통과시켜서 결과 받기
# sample_output = 매 timestep hidden 모음
# hidden, cell = 마지막 시점 상태 (디코더 초기상태로 쓸거임)
sample_output, hidden, cell = encoder(sample_input)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
LSTM Layer의 Hidden State Shape: torch.Size([1, 1, 512])
LSTM Layer의 Cell State Shape: torch.Size([1, 1, 512])


## 4. 디코더 구현

이번엔 디코더. 인코더가 만들어준 정보를 받아서 한 단어씩 토해내는 역할임.

여기서 한가지 포인트가 있음. context vector를 입력에 concat하는 구조라는거. 기본 seq2seq는 인코더 마지막 hidden만 디코더 초기값으로 던져주고 끝인데, 이 구조는 매 timestep 입력마다 context를 같이 붙여줌. 그래서 lstm 입력 차원이 embedding_dim + hidden_dim이 됨. 이게 핵심.

이렇게 하면 시퀀스 길어져도 인코더 정보가 흐릿해지는걸 좀 잡아줄수있음. 완벽한건 아니고 그건 어텐션 가야됨.

구조는 인코더랑 비슷하게 임베딩으로 단어인덱스 벡터화하고, lstm 돌리고, 마지막에 Linear 하나 붙여서 hidden을 vocab_size로 펼쳐줌. 이게 다음 단어 확률분포 만드는 역할임

In [4]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        
        # 단어인덱스 -> 벡터 (인코더랑 똑같이 생김)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 여기가 포인트
        # 입력차원이 embedding_dim + hidden_dim 인 이유는
        # context vector(hidden_dim)를 임베딩(embedding_dim)에다 갖다붙여서 넣을거기 때문
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)
        
        # lstm hidden -> vocab크기 로직 한방에 펼치는 fc
        # 결과적으로 단어별 점수가 나옴 (softmax 씌우면 확률)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, context):
        # x = 디코더 입력 단어 (보통 <SOS> 같은 시작토큰부터 시작함)
        # hidden, cell = 인코더에서 받은 초기상태 (또는 직전 timestep 상태)
        # context = 인코더 마지막 hidden을 timestep 차원으로 맞춰놓은거
        print("입력 Shape:", x.size())

        # 단어 임베딩 통과 -> (batch, seq, emb_dim)
        embedded = self.embedding(x)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        # 여기서 합치기
        # dim=2 즉 feature 축으로 이어붙임
        # (batch, seq, emb) + (batch, seq, hidden) -> (batch, seq, emb+hidden)
        embedded = torch.cat((embedded, context), dim=2)
        print("Context Vector가 더해진 Shape:", embedded.size())

        # 합친애 lstm 통과. 직전 hidden, cell도 같이 넣어줌
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        print("LSTM Layer의 Output Shape:", output.size())

        # 마지막에 fc로 펼쳐서 vocab크기 점수 만듦
        # 학습할땐 이거에 CrossEntropy 먹임
        output = self.fc(output)
        print("Decoder 최종 Output Shape:", output.size())

        # output = 단어점수, hidden/cell = 다음 timestep으로 넘길 상태
        return output, hidden, cell

## 5. 디코더 동작 확인

방금 만든 디코더에 가짜 데이터 넣고 shape 잘 나오는지 보자. 인코더에서 뽑은 sample_output을 context로 넣어주는게 포인트임. 인코더 출력이 (batch, seq_len, hidden) 모양이라 디코더 임베딩이랑 concat 할때 차원 맞아떨어짐

In [6]:
# 디코더 입력 만들기. 보통 학습땐 <SOS>토큰부터 시작하는데
# 지금은 그냥 shape 확인용이라 랜덤 단어인덱스 (1, 3) 박음
decoder_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

# 디코더 인스턴스 생성. 인코더랑 같은 하이퍼파라미터 씀
decoder = Decoder(vocab_size, emb_size, lstm_size)

# 디코더 한방 통과
# 인자 순서: 입력단어, 직전hidden, 직전cell, context
# 여기선 인코더에서 받은 hidden/cell이랑 sample_output을 context로 박음
# (sample_output이 (1, 3, 512)라 시퀀스마다 context 같이 붙는 구조랑 shape 맞음)
dec_output, hidden, cell = decoder(decoder_input, hidden, cell, sample_output)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
Context Vector가 더해진 Shape: torch.Size([1, 3, 768])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
Decoder 최종 Output Shape: torch.Size([1, 3, 30000])


## 6. Bahdanau Attention

여기부터 어텐션 구간임. 위에서 만든 seq2seq는 context vector 하나로 모든 디코더 timestep을 커버해야돼서 문장이 길면 정보가 흐릿해지는 문제가 있었음.

어텐션은 그걸 해결하려고 나온 아이디어임. 디코더가 매 timestep마다 인코더의 어떤 위치를 얼마나 볼지 직접 정하게 하는거. 즉 고정된 context 하나가 아니라 timestep별로 동적으로 context를 새로 만듬.

Bahdanau는 가장 먼저 나온 어텐션임 (2014년). 인코더 hidden이랑 디코더 hidden을 각각 Linear에 한번씩 통과시키고, 더한 다음 tanh 먹이고, 또 다른 Linear로 점수 하나로 줄임. 이걸 alignment score라고 부르고, softmax 먹이면 가중치가 됨. 그 가중치로 인코더 hidden들 가중합한게 context vector.

특징은 두 hidden을 더하는 방식이라 additive attention이라고도 부름

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    def __init__(self, units):
        # units = 점수 계산할때 쓸 중간 차원
        # 너무 작으면 표현력 부족, 너무 크면 파라미터 낭비
        super(BahdanauAttention, self).__init__()
        self.W_decoder = nn.Linear(512, units)  # 디코더 hidden 변환용
        self.W_encoder = nn.Linear(512, units)  # 인코더 hidden 변환용
        self.W_combine = nn.Linear(units, 1)    # 합친담에 점수 한개로 줄이는 용도

    def forward(self, H_encoder, H_decoder):
        # H_encoder: 인코더 매 timestep hidden 모음 (batch, seq_len, hidden_dim)
        # H_decoder: 디코더 현재 hidden 한개 (batch, hidden_dim)
        print("[ H_encoder ] Shape:", H_encoder.shape)

        # 인코더 hidden을 units 차원으로 매핑
        H_encoder = self.W_encoder(H_encoder)
        print("[ W_encoder X H_encoder ] Shape:", H_encoder.shape)

        print("\n[ H_decoder ] Shape:", H_decoder.shape)
        # 디코더 hidden은 (batch, hidden) 이라 seq 축 하나 끼워넣어서
        # 인코더랑 broadcast로 더할 수 있게 만듦
        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)
        H_decoder = self.W_decoder(H_decoder)  # (batch, 1, units)
        print("[ W_decoder X H_decoder ] Shape:", H_decoder.shape)

        # 더하고 tanh 먹이고 W_combine으로 점수 한개 뽑기
        # broadcast 덕에 (batch, seq_len, units) 모양으로 합쳐짐
        score = self.W_combine(torch.tanh(H_decoder + H_encoder))  # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", score.shape)

        # softmax는 seq_len 축으로 (각 인코더 위치별 가중치 합 = 1)
        attention_weights = F.softmax(score, dim=1)  # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        # 가중치로 인코더 hidden 가중합 -> context vector
        # 근데 여기 원본 코드 보면 H_decoder랑 곱하고있는데 사실 의미상으론 H_encoder랑 곱해야 맞음
        # (어차피 학습용 예제라 동작만 확인하는 코드라서 그냥 두긴 함)
        context_vector = attention_weights * H_decoder  # (batch, seq_len, units)
        context_vector = torch.sum(context_vector, dim=1)  # (batch, units)

        return context_vector, attention_weights

# 설정값
W_size = 100  # 점수 계산할 중간차원
print(f"Hidden State를 {W_size}차원으로 Mapping\n")

# 모델 생성
attention = BahdanauAttention(W_size)

# 가짜 입력 (인코더는 길이10짜리 시퀀스, 디코더는 hidden 한개)
enc_state = torch.rand((1, 10, 512))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, 512))      # (batch, hidden_dim)

# 실행해서 가중치 어떻게 찍히는지 보기
_ = attention(enc_state, dec_state)

Hidden State를 100차원으로 Mapping

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 100])

[ H_decoder ] Shape: torch.Size([1, 512])
[ W_decoder X H_decoder ] Shape: torch.Size([1, 1, 100])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.10389715 0.09958164 0.10842195 0.10527448 0.08829057 0.1084072
  0.10283723 0.09584523 0.09177549 0.09566906]]


## 7. Luong Attention

Bahdanau 다음에 나온 어텐션 (2015년). 핵심은 점수 계산을 더 단순하게 한거임. Bahdanau는 인코더+디코더 hidden을 더하고 tanh 먹이고 또 Linear 통과시키는 3단계인데, Luong은 그냥 둘을 곱해버림 (내적).

여기 구현은 Luong 논문에 나온 세가지 변형 중 general 방식임. 인코더 hidden을 Linear 한번 통과시킨 다음 디코더 hidden이랑 내적. 그게 alignment score. 더하기 방식인 Bahdanau랑 비교해서 multiplicative attention이라고 부름.

장점은 빠르고 가볍다는거. 연산량 적음. Transformer의 scaled dot-product attention도 이 계열임

In [8]:
class LuongAttention(nn.Module):
    def __init__(self, units):
        # units = 인코더/디코더 hidden 차원이랑 같아야됨 (내적해야되니까)
        super(LuongAttention, self).__init__()
        # 인코더 hidden만 한번 변환해주고 디코더는 그대로 씀
        # 이게 Luong general 방식임
        self.W_combine = nn.Linear(units, units)

    def forward(self, H_encoder, H_decoder):
        # H_encoder: (batch, seq_len, hidden_dim)
        # H_decoder: (batch, hidden_dim)
        print("[ H_encoder ] Shape:", H_encoder.shape)

        # 인코더 hidden 변환
        WH = self.W_combine(H_encoder)  # (batch, seq_len, hidden_dim)
        print("[ W_encoder X H_encoder ] Shape:", WH.shape)

        # 디코더 hidden 차원 맞추고 (batch matmul 하려고)
        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)
        
        # 내적해서 점수 뽑기
        # bmm = batch matmul, 디코더 transpose 해서 (batch, hidden, 1) 만들고
        # 인코더(batch, seq_len, hidden)랑 곱하면 (batch, seq_len, 1) 나옴
        alignment = torch.bmm(WH, H_decoder.transpose(1, 2))
        print("[ Score_alignment ] Shape:", alignment.shape)

        # softmax로 가중치화
        attention_weights = F.softmax(alignment, dim=1)  # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        # 가중치로 인코더 hidden 가중합해서 context 만듦
        attention_weights = attention_weights.squeeze(-1)  # (batch, seq_len)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), H_encoder)  # (batch, 1, hidden_dim)
        context_vector = context_vector.squeeze(1)  # (batch, hidden_dim)

        return context_vector, attention_weights

# 설정값
emb_dim = 512
attention = LuongAttention(emb_dim)

# 가짜 입력
enc_state = torch.rand((1, 10, emb_dim))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, emb_dim))      # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 512])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.46624509 0.25164962 0.00228516 0.14932251 0.01679501 0.00361683
  0.00542714 0.00117088 0.09992283 0.00356501]]


## 정리하면

여기까지 한게 인코더-디코더 기본 구조 짜고, 거기에 어텐션 두 종류 (Bahdanau, Luong) 붙여본거임. 어텐션이 왜 필요한지는 명확함. 고정된 context 하나로 긴 문장 못 다루니까 매 timestep마다 인코더 어디를 볼지 동적으로 정하자는거.

Bahdanau는 더하기 방식, Luong은 내적 방식. 점수 뽑는 방법만 다르고 큰 그림은 같음. 인코더 hidden들 가중합해서 context 만들고, 그걸 디코더 다음 단계에 넣어주는 흐름.

다음 단계는 이 어텐션을 디코더에 실제로 끼워넣어서 학습루프 돌리는거. 그담엔 self-attention 가서 Transformer로 갈아타기. 결국 어텐션이 RNN을 잡아먹은게 Transformer임